<a href="https://colab.research.google.com/github/Abir-world/Image-processing/blob/main/Copy_of_DFT_and_IDFT_Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DFT and IDFT

A very simple complex problem using python. In this context, we applied FT

In [ ]:
import numpy as np

f = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

M, N = f.shape
F = np.zeros((M, N), dtype=complex)

for u in range(M):
  for v in range(N):
    s = 0.0 + 0.0j
    for x in range(M):
      for y in range(N):
        s += f[x,y] * np.exp(-2j*np.pi*((u*x)/M + (v*y)/N))
    F[u,v] = s

print(F)

In [ ]:
M, N = F.shape

f_reconstructed = np.zeros((M, N), dtype=complex)

for x in range(M):
  for y in range(N):
    s = 0.0 + 0.0j
    for u in range(M):
      for v in range(N):
        s += F[u, v] * np.exp(2j*np.pi*((u*x)/M + (v*y)/N))

    f_reconstructed[x, y] = s / (M * N)

print(np.round(f_reconstructed.real, 10))

# More Convenient Way

In [ ]:
# forward way--> special domain to frequency domain
F = np.fft.fft2(f)
print(f"Fourier Result:\n {F}")

print()

# shifting property (apply when we dealing with real image)

# In the DFT, the zero frequency F(0,0) corresponds to the DC component — the average brightness of the image.
# By default, np.fft.fft2(f), places this DC component at the top-left corner of the frequency matrix.
# The low frequencies (smooth variations) are at the corners.
# The high frequencies (edges and details) are in the center of the matrix.
# Low frequencies → center
# High frequencies → edges
# Shift: Moves the zero frequency (DC) to the center of the spectrum.
#


F_shifted = np.fft.fftshift(F)
print(f"Shift: \n{F_shifted}")

# print()

# # backword way--> frequency domain to special domain

f = np.fft.ifft2(F)
print(f"Original Result:\n {np.real(f)}")

# For Image

In [ ]:
import matplotlib.pyplot as plt
def show_images(image, title: str=""):
  plt.imshow(image, cmap='gray')
  plt.title(title)
  plt.axis('off')
  plt.show()

In [ ]:
import cv2
import numpy as np



# load image
from google.colab import files
img_file = files.upload()

In [ ]:
img_bgr = cv2.imread(filename='/content/clean_image.png')
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

In [ ]:
print(img.shape)

In [ ]:
show_images(img, "Orginal Clean Image")

In [ ]:
# Add Gaussian Noise
mean = 0
sigma = 25

noise = np.random.normal(mean, sigma, img.shape)

noisy_img = img.astype(np.float32) + noise
noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)

show_images(noisy_img, "Gaussian Noise Image")

In [ ]:
# apply DFT
F = np.fft.fft2(noisy_img)
magnitude = np.log(1 + np.abs(F))  # |F(u,v)|
show_images(magnitude, "DFT Image (Magnitude Spectrum)")

# # apply shifting
# img_shift = np.fft.fftshift(img_dft)


In [ ]:
# apply shift
F_shift = np.fft.fftshift(F)
magnitude = np.log(1 + np.abs(F_shift))  # |F(u,v)|
show_images(magnitude, "DFT Shift Image")

In [ ]:
# Create Gaussian Low-Pass Filter

rows, cols = img.shape
crow, ccol = rows // 2, cols // 2

"""
D0	Result
20	Very blurry
50	Blurry
100	Moderate smoothing
150	Light smoothing
250	Almost original
"""

D0 = 20     # Cutoff Frequency

H = np.zeros((rows, cols))

for u in range(rows):
  for v in range(cols):
    D = np.sqrt((u-crow)**2 + (v-ccol)**2)
    H[u,v] = np.exp(-(D**2)/(2*(D0**2)))


# Apply Gaussian Filter
G_shift = F_shift * H

# show image after apply filter
magnitude = np.log(1 + np.abs(G_shift))  # |F(u,v)|
show_images(magnitude, "DFT Shift Image")

In [ ]:
# Apply IDFT to get back the original image
G = np.fft.ifftshift(G_shift)

reconstructed = np.fft.ifft2(G)
reconstructed = np.abs(reconstructed)
reconstructed = np.clip(reconstructed, 0, 255)
reconstructed = reconstructed.astype(np.uint8)

show_images(reconstructed, "Reconstructed Image")